# 02 — Recode Nonpurchase and Construct Trip-Level Food Spending

**Healthy Purchase Cycle | 2008–2017 | Python + DuckDB**

This notebook preserves each original purchase line, validates Magnet-to-original purchase multiplicities before anti-joining, and writes year-partitioned Parquet plus audit CSVs. Run after the beverage upstream correction and Step 01 merge have been completed. Original category outcomes 1/2 (and cereal 3) remain unchanged; category nonpurchase is coded 0.


In [ ]:
from pathlib import Path
import duckdb, pandas as pd, time, shutil

REPO = Path(r"C:/Users/rwen13/Documents/healthy_purchase_cycle")
DATA = Path(r"C:/Users/rwen13/Documents/healthy_purchase_data")
RAW_HMS = Path(r"C:/Users/rwen13/Documents/2004-2020_Panel_Data/nielsen_extracts/HMS")
FOOD_LIST = REPO / "data_cleaning/upc_classification/food_list.xlsx"
MASTER = RAW_HMS / "Master_Files/Latest/products.tsv"
SOURCE_PANEL = DATA / "processed/hmm_panel/trip_purchase_panel"
CORRECTED_PANEL = DATA / "processed/hmm_panel/trip_purchase_panel_y0"
FOOD_PANEL = DATA / "processed/hmm_panel/trip_food_spending"
STAGE = DATA / "intermediate/hmm_food_spending_stage"
AUDIT = DATA / "processed/hmm_panel/validation"
TEMP = DATA / "temp/duckdb_food_spending"
MAGNET_DIR = DATA / "intermediate/magnet_food_purchases"
YEARS = range(2008, 2018)
SOURCE_YEARS = range(2008, 2019)   # 2018 source may contain 2017 purchases
MEMORY_LIMIT, THREADS = '4GB', 4
# Match the confirmed upstream price convention: max(total_price_paid - coupon_value, 0)
# If an original category script uses another convention, compare its audit before using dollar shares.
USE_CAPPED_PRICE = True
# Allow cross-source trip-ID formats only when validated against trip-match audits below.

for folder in [CORRECTED_PANEL, FOOD_PANEL, STAGE, AUDIT, TEMP]:
    folder.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(database=':memory:')
con.execute(f"SET memory_limit='{MEMORY_LIMIT}'")
con.execute(f"SET threads={THREADS}")
con.execute(f"SET temp_directory='{TEMP.as_posix()}'")

def qpath(p):
    return "'" + Path(p).as_posix().replace("'", "''") + "'"
def qname(s):
    return '"' + s.replace('"','""') + '"'
def parquet_ref(path):
    return f'read_parquet({qpath(path)})'
def raw_ref(path):
    return f"read_csv({qpath(path)}, delim='\\t', header=true, all_varchar=true)"

assert FOOD_LIST.is_file(), FOOD_LIST
assert MASTER.is_file(), MASTER
print('Ready: DuckDB', duckdb.__version__)

## A. Correct category nonpurchase outcomes

This reads each existing `trips.parquet` separately, checks that the six-category outcome values are recognized, writes a **new file** with `NULL → 0` (category not represented in the category-merge input), and audits the number changed. It does not recompute the six-category classifications. In particular, `cereal_y = 0` means **no classified cereal purchase**, not proof of no cereal at all.

In [ ]:
OUTCOMES = ['fv_y','milk_y','yogurt_y','cereal_y','snack_y','beverage_y']
SPENDS = ['fresh_spending','non_fresh_spending','skimmed_milk_spending',
          'low_fat_milk_spending','other_milk_spending',
          'skimmed_yogurt_spending','low_fat_yogurt_spending','other_yogurt_spending',
          'low_sugar_cereal_spending','medium_sugar_cereal_spending','high_sugar_cereal_spending',
          'healthy_snack_spending','unhealthy_snack_spending',
          'healthy_beverage_spending','unhealthy_beverage_spending']
expected = {'fv_y':{1,2},'milk_y':{1,2},'yogurt_y':{1,2},
            'cereal_y':{1,2,3},'snack_y':{1,2},'beverage_y':{1,2}}
recode_audit=[]
for year in YEARS:
    source = SOURCE_PANEL / f'panel_year={year}' / 'trips.parquet'
    assert source.is_file(), f'Run Step 01 first: {source}'
    rel=parquet_ref(source)
    cols = set(con.execute(f'DESCRIBE SELECT * FROM {rel}').df()['column_name'])
    required = set(['hhid','panel_year','trip_id','purchase_date',*OUTCOMES,*SPENDS])
    assert required <= cols, f'Missing columns: {sorted(required - cols)}'
    checks = ', '.join(f"COUNT(*) FILTER (WHERE {qname(y)} IS NOT NULL AND {qname(y)} NOT IN ({','.join(map(str,sorted(expected[y])))})) AS {qname(y)}" for y in OUTCOMES)
    invalid = con.execute(f'SELECT {checks} FROM {rel}').fetchone()
    if any(invalid):
        raise ValueError(f'{year} has unexpected non-null outcome codes: {dict(zip(OUTCOMES, invalid))}')
    nulls = con.execute('SELECT '+ ', '.join(f'COUNT(*) FILTER (WHERE {qname(y)} IS NULL)' for y in OUTCOMES)+f' FROM {rel}').fetchone()
    dst= CORRECTED_PANEL / f'panel_year={year}' / 'trips.parquet'
    dst.parent.mkdir(parents=True,exist_ok=True)
    select_expr = []
    for field in con.execute(f'DESCRIBE SELECT * FROM {rel}').df()['column_name']:
        if field in OUTCOMES:
            select_expr.append(f'COALESCE({qname(field)}, 0)::INTEGER AS {qname(field)}')
        elif field in SPENDS:
            select_expr.append(f'COALESCE({qname(field)}, 0)::DOUBLE AS {qname(field)}')
        else:
            select_expr.append(qname(field))
    con.execute(f"COPY (SELECT {', '.join(select_expr)} FROM {rel}) TO {qpath(dst)} (FORMAT PARQUET, COMPRESSION ZSTD)")
    row={'panel_year':year,'trips':con.execute(f'SELECT COUNT(*) FROM {rel}').fetchone()[0]}
    row.update({f'{y}_null_to_zero':n for y,n in zip(OUTCOMES,nulls)})
    recode_audit.append(row)
pd.DataFrame(recode_audit).to_csv(AUDIT/'category_outcome_zero_recode.csv',index=False)
display(pd.DataFrame(recode_audit))

## B. Load verified food-module list and build food UPC lookup

Match regular foods using the **complete** Excel `product_module_code` list. Product UPC and version stay strings. Normalize leading zeros for **module codes only**. Flag ambiguous master-product UPC/version rows instead of arbitrarily selecting one. The Magnet output already contains the product-matching rules (reference cards and codes 453/460); it is not reclassified here.

In [ ]:
food_list=pd.read_excel(FOOD_LIST, dtype={'product_module_code':'string'})
assert 'product_module_code' in food_list.columns
food_list['module']=(food_list['product_module_code'].astype('string').str.strip()
    .str.replace(r'\\.0+$','',regex=True).str.replace(r'^0+(?=\\d)', '',regex=True))
modules=sorted(set(food_list['module'].dropna())-{'','<NA>'})
assert modules, 'Empty food-module list'
con.register('food_module_df',pd.DataFrame({'module':modules}))
con.execute('CREATE OR REPLACE TEMP TABLE food_modules AS SELECT DISTINCT module FROM food_module_df')
print('Allowed food modules:',len(modules))
master_cols=set(con.execute(f'DESCRIBE SELECT * FROM {raw_ref(MASTER)}').df()['column_name'])
for req in ['upc','upc_ver_uc','product_module_code']:
    assert req in master_cols, f'Master missing {req}'
con.execute(f"""
 CREATE OR REPLACE TEMP TABLE product_module_lookup AS
 SELECT TRIM(upc) AS upc, TRIM(upc_ver_uc) AS upc_ver_uc,
        CASE WHEN ltrim(regexp_replace(TRIM(product_module_code), '\\.0+$',''), '0') = ''
             THEN '0' ELSE ltrim(regexp_replace(TRIM(product_module_code), '\\.0+$',''), '0') END AS module
 FROM {raw_ref(MASTER)}
 WHERE upc IS NOT NULL AND upc_ver_uc IS NOT NULL
""")
conflicts=con.execute('''SELECT COUNT(*) FROM (
 SELECT upc,upc_ver_uc FROM product_module_lookup
 GROUP BY 1,2 HAVING COUNT(DISTINCT module)>1)''').fetchone()[0]
if conflicts:
    raise ValueError(f'Master has {conflicts} UPC-version keys with conflicting modules; resolve before aggregation')
con.execute('''CREATE OR REPLACE TEMP TABLE food_product_lookup AS
 SELECT DISTINCT p.upc,p.upc_ver_uc FROM product_module_lookup p
 INNER JOIN food_modules m ON p.module=m.module''')
print('Food UPC-versions:',con.execute('SELECT COUNT(*) FROM food_product_lookup').fetchone()[0])

## C. Stage actual purchase spending, one source year at a time

Regular food is identified by `food_list.xlsx` and original purchases. Magnet food comes from previously extracted Magnet purchase lines. Repeated trip/UPC/version keys are **retained**, not deduplicated. Before excluding Magnet keys from the regular-food branch, the notebook verifies that the number of Magnet lines equals the number of original purchase lines for **every** matching key. A mismatch stops the calculation and prints examples. Matching counts support key-level exclusion but do not independently verify amounts.


In [ ]:
# Run this inexpensive schema/preflight audit before launching the yearly scans.
for sy in SOURCE_YEARS:
    base=RAW_HMS/str(sy)/'Annual_Files'
    for label,name in [('purchases',f'purchases_{sy}.tsv'),('trips',f'trips_{sy}.tsv')]:
        assert (base/name).is_file(), f'Missing {label}: {base/name}'
    magnet=MAGNET_DIR/f'magnet_food_purchases_{sy}.tsv'
    assert magnet.is_file(), f'Missing extracted magnet file: {magnet}'
print('All source-year purchase, trip and Magnet files found.')
for label, path, needed in [
 ('purchases',RAW_HMS/'2008/Annual_Files/purchases_2008.tsv', ['upc','upc_ver_uc','trip_code_uc','total_price_paid','coupon_value']),
 ('trips',RAW_HMS/'2008/Annual_Files/trips_2008.tsv',['trip_code_uc','household_code','purchase_date']),
 ('magnet',MAGNET_DIR/'magnet_food_purchases_2008.tsv',['upc','upc_ver_uc','trip_code_uc','magnet_food_spend_increment_capped','purchase_date'])]:
    cols=set(con.execute(f'DESCRIBE SELECT * FROM {raw_ref(path)}').df()['column_name'])
    assert set(needed)<=cols, f'{label}: missing {set(needed)-cols}'
    print(label,'columns OK')

In [ ]:
# Each source-year file is processed separately; DuckDB executes file scans on disk.
# Intermediate Parquet contains only the records needed for analysis years.
source_audit=[]
for sy in SOURCE_YEARS:
    started=time.perf_counter()
    base=RAW_HMS/str(sy)/'Annual_Files'
    p=raw_ref(base/f'purchases_{sy}.tsv')
    t=raw_ref(base/f'trips_{sy}.tsv')
    m=raw_ref(MAGNET_DIR/f'magnet_food_purchases_{sy}.tsv')
    stage_dir=STAGE/f'source_year={sy}'
    stage_dir.mkdir(parents=True,exist_ok=True)
    # Treat 2018 source-year boundary dates like all other purchase dates.
    con.execute(f"""CREATE OR REPLACE TEMP VIEW year_trips AS
        SELECT TRIM(trip_code_uc) AS trip_code_uc,
               TRIM(household_code) AS hhid,
               TRY_CAST(purchase_date AS DATE) AS purchase_date
        FROM {t}""")
    dup_trip=con.execute('SELECT COUNT(*) FROM (SELECT trip_code_uc FROM year_trips GROUP BY 1 HAVING COUNT(*)>1)').fetchone()[0]
    if dup_trip: raise ValueError(f'{sy}: {dup_trip} duplicate trip_code_uc in trips source')
    # Materialize Magnet keys and amounts for a relatively small source-year output.
    con.execute(f"""CREATE OR REPLACE TEMP TABLE year_magnet AS
        SELECT TRIM(trip_code_uc) trip_code_uc, TRIM(upc) upc,
               TRIM(upc_ver_uc) upc_ver_uc,
               TRY_CAST(purchase_date AS DATE) purchase_date,
               TRIM(household_code) hhid,
               TRY_CAST(magnet_food_spend_increment_capped AS DOUBLE) amount
        FROM {m}""")
    # Repeated trip/UPC/version combinations are valid purchase lines.
    # Never deduplicate them: amounts from every row are required.
    # Compare purchase-line multiplicities against the raw source instead.
    con.execute("""CREATE OR REPLACE TEMP TABLE year_magnet_keys AS
      SELECT trip_code_uc, upc, upc_ver_uc, COUNT(*) AS magnet_rows
      FROM year_magnet
      GROUP BY 1,2,3""")
    magnet_dups=con.execute("""SELECT COUNT(*) FROM year_magnet_keys
      WHERE magnet_rows > 1""").fetchone()[0]

    # Read original purchase lines with original UPC keys preserved as strings.
    con.execute(f"""CREATE OR REPLACE TEMP VIEW year_purchases AS
      SELECT TRIM(trip_code_uc) trip_code_uc, TRIM(upc) upc,
             TRIM(upc_ver_uc) upc_ver_uc,
             TRY_CAST(total_price_paid AS DOUBLE) total_price_paid,
             COALESCE(TRY_CAST(coupon_value AS DOUBLE),0) coupon_value
      FROM {p}""")

    # SEMI JOIN prevents repeated Magnet keys from multiplying purchase rows.
    # A key-level anti-join is valid only if every Magnet purchase line
    # corresponds to exactly one raw purchase line with the same key.
    con.execute("""CREATE OR REPLACE TEMP TABLE year_raw_magnet_key_counts AS
      SELECT p.trip_code_uc, p.upc, p.upc_ver_uc,
             COUNT(*) AS purchase_rows
      FROM year_purchases p
      SEMI JOIN year_magnet_keys m
        USING (trip_code_uc, upc, upc_ver_uc)
      GROUP BY 1,2,3""")
    overlap=con.execute("""SELECT
        COUNT(*) FILTER (WHERE p.purchase_rows IS NULL) AS unmatched_keys,
        COUNT(*) FILTER (WHERE p.purchase_rows IS NOT NULL
                        AND p.purchase_rows <> m.magnet_rows) AS unequal_keys,
        COALESCE(SUM(m.magnet_rows),0) AS magnet_rows,
        COALESCE(SUM(p.purchase_rows),0) AS matched_raw_rows
      FROM year_magnet_keys m LEFT JOIN year_raw_magnet_key_counts p
        USING (trip_code_uc, upc, upc_ver_uc)""").fetchone()
    unmatched_keys, unequal_keys, magnet_rows, raw_rows = overlap
    print(f'{sy}: repeated Magnet trip/UPC keys={magnet_dups:,}; '
          f'unmatched={unmatched_keys:,}; unequal multiplicities={unequal_keys:,}')
    if unmatched_keys or unequal_keys:
        samples=con.execute("""SELECT m.trip_code_uc,m.upc,m.upc_ver_uc,
                  m.magnet_rows,p.purchase_rows
            FROM year_magnet_keys m LEFT JOIN year_raw_magnet_key_counts p
              USING (trip_code_uc,upc,upc_ver_uc)
            WHERE p.purchase_rows IS NULL OR m.magnet_rows <> p.purchase_rows
            LIMIT 15""").df()
        display(samples)
        raise ValueError(f'{sy}: Magnet/raw purchase-line counts mismatch '
                         f'(unmatched={unmatched_keys}, unequal={unequal_keys}). '
                         'Cannot safely apply key-level anti-join.')
    # All same-key raw rows are accounted for in Magnet; exclude that
    # key from regular foods and add all Magnet increments exactly once.
    ambig = unequal_keys
    # Matched regular-food purchases, excluding corresponding Magnet keys.
    out_regular=stage_dir/'regular.parquet'
    con.execute(f"""COPY (
      SELECT tr.hhid, tr.purchase_date, tr.trip_code_uc,
             CAST(GREATEST(COALESCE(p.total_price_paid - p.coupon_value,0),0) AS DOUBLE) AS food_amount
      FROM year_purchases p
      INNER JOIN food_product_lookup f USING (upc, upc_ver_uc)
      INNER JOIN year_trips tr USING (trip_code_uc)
      WHERE tr.hhid IS NOT NULL AND tr.purchase_date BETWEEN DATE '2008-01-01' AND DATE '2017-12-31'
        AND NOT EXISTS (
          SELECT 1 FROM year_magnet_keys m
          WHERE m.trip_code_uc=p.trip_code_uc AND m.upc=p.upc AND m.upc_ver_uc=p.upc_ver_uc
        )
    ) TO {qpath(out_regular)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    out_magnet=stage_dir/'magnet.parquet'
    con.execute(f"""COPY (
      SELECT hhid, purchase_date, trip_code_uc,
             CAST(COALESCE(amount,0) AS DOUBLE) AS food_amount
      FROM year_magnet
      WHERE hhid IS NOT NULL AND purchase_date BETWEEN DATE '2008-01-01' AND DATE '2017-12-31'
    ) TO {qpath(out_magnet)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    regular_n=con.execute(f'SELECT COUNT(*) FROM {parquet_ref(out_regular)}').fetchone()[0]
    magnet_n=con.execute(f'SELECT COUNT(*) FROM {parquet_ref(out_magnet)}').fetchone()[0]
    source_audit.append({'source_year':sy,'regular_food_purchase_rows':regular_n,
        'magnet_food_purchase_rows':magnet_n,'duplicate_magnet_keys':magnet_dups,
        'unmatched_magnet_keys':unmatched_keys,
        'unequal_row_count_keys':unequal_keys,'magnet_rows_reconciled':magnet_rows,
        'raw_rows_reconciled':raw_rows,'seconds':round(time.perf_counter()-started,1)})
    print(f'{sy}: regular={regular_n:,}, magnet={magnet_n:,}')
pd.DataFrame(source_audit).to_csv(AUDIT/'food_spending_source_audit.csv',index=False)
display(pd.DataFrame(source_audit))

## D. Sum food dollars by actual trip; match against the corrected six-category panel

A source-year trip code is first joined to its household and actual date. The corrected category panel contains `trip_id`, which is expected to correspond to Nielsen `trip_code_uc`. We **measure** the match rate rather than silently assuming the join is correct. If trip codes are not equivalent, stop and inspect the audit before proceeding.

We deliberately do **not** create `food_spend = 0` for an unmatched category trip, because zero food purchases and a failed source match are different.

In [ ]:
# Build per-year totals from staged source-year files using DuckDB wildcard paths.
# Group before joining: never join purchase rows directly to the trip panel.
regular_files=[STAGE/f'source_year={sy}'/'regular.parquet' for sy in SOURCE_YEARS]
magnet_files=[STAGE/f'source_year={sy}'/'magnet.parquet' for sy in SOURCE_YEARS]
def rel_for(paths):
    return 'read_parquet(['+', '.join(qpath(p) for p in paths)+'])'
regular_rel=rel_for(regular_files)
magnet_rel=rel_for(magnet_files)
food_audit=[]
for year in YEARS:
    t0=time.perf_counter()
    panel=CORRECTED_PANEL/f'panel_year={year}'/'trips.parquet'
    assert panel.is_file()
    # Stage annual totals so the final join is small and deterministic.
    food_totals=STAGE/f'food_trip_totals_{year}.parquet'
    con.execute(f"""COPY (
        SELECT hhid, purchase_date, trip_code_uc,
               SUM(regular_amount) AS regular_food_spend,
               SUM(magnet_amount) AS magnet_food_spend,
               SUM(regular_amount+magnet_amount) AS food_spend
        FROM (
          SELECT hhid,purchase_date,trip_code_uc,food_amount AS regular_amount,0::DOUBLE AS magnet_amount
          FROM {regular_rel} WHERE YEAR(purchase_date)={year}
          UNION ALL
          SELECT hhid,purchase_date,trip_code_uc,0::DOUBLE,food_amount
          FROM {magnet_rel} WHERE YEAR(purchase_date)={year}
        ) z GROUP BY 1,2,3
    ) TO {qpath(food_totals)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    f=parquet_ref(food_totals); c=parquet_ref(panel)
    dup=con.execute(f'''SELECT COUNT(*) FROM (SELECT hhid,purchase_date,trip_code_uc FROM {f}
      GROUP BY ALL HAVING COUNT(*)>1)''').fetchone()[0]
    assert dup==0, f'Duplicate food trip keys for {year}'
    dest=FOOD_PANEL/f'panel_year={year}'/'trips.parquet'
    dest.parent.mkdir(parents=True,exist_ok=True)
    con.execute(f"""COPY (
      SELECT c.hhid, c.panel_year, c.trip_id, c.purchase_date,
             f.regular_food_spend, f.magnet_food_spend, f.food_spend,
             f.trip_code_uc IS NOT NULL AS food_trip_matched
      FROM {c} c LEFT JOIN {f} f
        ON c.hhid=f.hhid AND c.purchase_date=f.purchase_date
           AND c.trip_id=f.trip_code_uc
    ) TO {qpath(dest)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    o=parquet_ref(dest)
    counts=con.execute(f'''SELECT COUNT(*) AS trips, COUNT(*) FILTER (WHERE food_trip_matched) AS matched,
      COUNT(*) FILTER (WHERE NOT food_trip_matched) AS unmatched,
      SUM(food_spend) AS food_dollars,
      COUNT(*) FILTER (WHERE food_spend < 0) AS negative_spend
      FROM {o}''').fetchone()
    row=dict(zip(['panel_year','matched','unmatched','food_dollars','negative_spend'],[year,*counts[1:]]))
    row['trips']=counts[0]; row['match_rate']=counts[1]/counts[0] if counts[0] else None
    row['seconds']=round(time.perf_counter()-t0,1)
    food_audit.append(row)
    print(f'{year}: {counts[1]:,}/{counts[0]:,} category trips matched to food source; rate {row["match_rate"]:.1%}')
pd.DataFrame(food_audit).to_csv(AUDIT/'trip_food_spending_merge_audit.csv',index=False)
display(pd.DataFrame(food_audit))

## E. Final safety checks and interpretation

- **Match coverage is critical.** This notebook does not silently convert unmatched trips to zero. Inspect `trip_food_spending_merge_audit.csv` before moving to Step 03.
- `food_spend` is independent of the six gated `y` values. Products bought on a healthy-outcome trip are *not* automatically classified as healthy spending.
- `food_spend` uses the specified purchase `total_price_paid - coupon_value`, capped at zero. This may differ from category spending conventions; validate before computing spending ratios.
- This construction targets **2008–2017 category-observed trips only**. The 2007 fresh-FV initialization uses a separate historical extraction and is not covered by this notebook.
- Food spending remains a separate trip table; Step 03 joins it to the HMM panel by `(hhid, panel_year, trip_id, purchase_date)`.
- For security and reproducibility, all intermediate outputs remain under local `healthy_purchase_data` (not GitHub).

In [ ]:
# Quick QA: validate nonnegative food dollars and final outcome recode.
for year in YEARS:
    food=parquet_ref(FOOD_PANEL/f'panel_year={year}'/'trips.parquet')
    panel=parquet_ref(CORRECTED_PANEL/f'panel_year={year}'/'trips.parquet')
    bad=con.execute(f'''SELECT COUNT(*) FROM {food}
        WHERE food_spend<0 OR regular_food_spend<0 OR magnet_food_spend<0''').fetchone()[0]
    assert bad==0, (year,'negative spending',bad)
    missing=con.execute('SELECT '+', '.join(f'COUNT(*) FILTER (WHERE {qname(y)} IS NULL)' for y in OUTCOMES)+f' FROM {panel}').fetchone()
    assert not any(missing), (year,'unexpected missing outcome',missing)
print('Outcome and spending checks passed. Review match-rate audit before downstream use.')